<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/28_interpolation/10_lagrange_interpolation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# This cell is for the Google Colaboratory
# https://stackoverflow.com/a/63519730
if 'google.colab' in str(get_ipython()):
  path_py = '/content/nmisp_py'

  import os
  if not os.path.exists(path_py):
    import subprocess
    subprocess.run(
        ('git', 'clone', 'https://github.com/kwlee2025cpp/nmisp_py')
    )
  assert os.path.exists(path_py)

  import sys
  sys.path.insert(0, path_py)


In [ ]:
# 그래프, 수학 기능 추가
# Add graph and math features
import matplotlib.pyplot as plt
import numpy as np


# 라그랑주 내삽<br>Lagrange Interpolation


앞에서는 두 점 사이를 직선으로 잇는 **선형 내삽** 을 살펴보았다. 그렇다면 점이 세 개, 다섯 개, 그 이상이면 어떨까? 모든 점을 동시에 지나는 다항식이 있을까? 어떻게 만들 수 있을까?<br>
The previous notebook covered **linear interpolation** &mdash; a straight line between two points. What if we have three, five, or more data points? Is there a single polynomial that passes through *every* one of them &mdash; and how do we build it?

라그랑주 내삽은 이 질문에 대한 가장 깔끔한 답이다. 핵심은 *기저 다항식* 이라는 작은 수학적 도구이다.<br>
Lagrange interpolation is the cleanest answer to that question. The key idea is a small but powerful gadget called *basis polynomials*.


## 라그랑주 기저 다항식<br>Lagrange Basis Polynomials


$n+1$ 개의 자료점 $(x_0, y_0), (x_1, y_1), \ldots, (x_n, y_n)$ 이 있다고 하자. 각 자료점 $j$ 마다 다음과 같은 다항식을 정의한다.<br>
Suppose we have $n+1$ data points $(x_0, y_0), \ldots, (x_n, y_n)$. For each index $j$, define the polynomial

$$
L_j(x) \;=\; \prod_{\substack{i=0\\ i \neq j}}^{n} \frac{x - x_i}{x_j - x_i}.
$$

이 식의 가장 중요한 성질은 다음과 같다.<br>
The crucial property of $L_j(x)$ is

$$
L_j(x_i) \;=\; \delta_{ij} \;=\; \begin{cases} 1 & i = j \\ 0 & i \neq j \end{cases}
$$

자기 자신의 자료점 $x_j$ 에서는 1, 다른 자료점 $x_i$ ($i \ne j$) 에서는 0 이 된다. 분자에 $(x - x_i)$ 인수가 들어 있기 때문이다.<br>
At its own node $x_j$ it is 1; at every other node $x_i$ it is 0, because $(x - x_i)$ is one of the factors in the numerator.

이 단순한 성질이 *바로* 우리에게 필요한 것이다. 자세한 의미는 다음 절에서 보게 된다.<br>
That tiny identity is *exactly* what we need &mdash; the next section makes this concrete.


In [ ]:
def lagrange_basis(x_nodes, j, x):
    """j 번째 라그랑주 기저 다항식 L_j(x) 의 값을 계산.
       Evaluate the j-th Lagrange basis polynomial at x."""
    x_nodes = np.asarray(x_nodes, dtype=float)
    x = np.asarray(x, dtype=float)
    n = len(x_nodes)
    result = np.ones_like(x)
    for i in range(n):
        if i == j:
            continue
        result = result * (x - x_nodes[i]) / (x_nodes[j] - x_nodes[i])
    return result


기저 다항식이 어떻게 생겼는지 그려 보자.<br>
Let's actually look at what these basis polynomials look like.


In [ ]:
x_nodes = np.linspace(-1.0, 1.0, 5)
x_dense = np.linspace(-1.0, 1.0, 400)

plt.figure(figsize=(8, 4.5))
for j in range(len(x_nodes)):
    plt.plot(x_dense, lagrange_basis(x_nodes, j, x_dense), label=f'$L_{j}(x)$')

# 자료점 위치 표시 / mark node positions
for x_n in x_nodes:
    plt.axvline(x_n, color='gray', linestyle=':', alpha=0.5)
plt.axhline(1.0, color='gray', linestyle=':', alpha=0.3)
plt.axhline(0.0, color='gray', linestyle=':', alpha=0.3)

plt.title('5개의 절점에 대한 라그랑주 기저 / Lagrange basis on 5 equispaced nodes')
plt.xlabel('$x$')
plt.ylabel('$L_j(x)$')
plt.legend(loc='upper center', ncol=5)
plt.grid(True)
plt.show()


각 곡선이 *자기* 절점에서 정확히 1 이고 *다른* 절점에서 정확히 0 인 것을 눈으로 확인할 수 있다.<br>
Each curve passes through height 1 at *its own* node and height 0 at every *other* node &mdash; visible by eye.


## 보간 다항식 만들기<br>Constructing the Interpolant


기저 다항식을 갖춘 다음, $y$ 값으로 가중합을 만든다.<br>
Once we have the basis polynomials, we just take a weighted sum with the $y$ values:

$$
p_n(x) \;=\; \sum_{j=0}^{n} y_j \, L_j(x).
$$

왜 이것이 모든 자료점을 지나는가? 어떤 절점 $x_i$ 에 대입해 보자.<br>
Why does this pass through every data point? Plug in any node $x_i$:

$$
p_n(x_i) \;=\; \sum_{j=0}^{n} y_j \, L_j(x_i) \;=\; \sum_{j=0}^{n} y_j \, \delta_{ij} \;=\; y_i.
$$

합 안의 모든 항이 0 이고 단 하나 $j = i$ 인 항만 $y_i \cdot 1 = y_i$ 가 된다. *기저 자체가 보간 조건을 자동으로 만족시킨다.*<br>
Every term in the sum vanishes except the single $j = i$ term, which is $y_i \cdot 1 = y_i$. *The basis itself enforces the interpolation conditions, automatically.*


In [ ]:
def lagrange_interpolant(x_nodes, y_nodes, x):
    """라그랑주 보간 다항식 p_n(x) 값.
       Evaluate the Lagrange interpolant p_n(x)."""
    x_nodes = np.asarray(x_nodes, dtype=float)
    y_nodes = np.asarray(y_nodes, dtype=float)
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    for j in range(len(x_nodes)):
        result = result + y_nodes[j] * lagrange_basis(x_nodes, j, x)
    return result


작은 예제로 확인해 보자: 4개의 자료점.<br>
A small sanity check on 4 data points.


In [ ]:
x_data = np.array([0.0, 1.0, 2.0, 4.0])
y_data = np.array([1.0, 2.0, 0.5, 3.0])

x_dense = np.linspace(x_data.min() - 0.5, x_data.max() + 0.5, 400)
y_dense = lagrange_interpolant(x_data, y_data, x_dense)

plt.figure(figsize=(8, 4.5))
plt.plot(x_dense, y_dense, label='Lagrange $p_3(x)$')
plt.plot(x_data, y_data, 'ko', markersize=8, label='data points')
plt.title('네 점을 지나는 3차 라그랑주 다항식 / Cubic Lagrange polynomial through 4 points')
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.legend(); plt.grid(True)
plt.show()

# 자료점에서 정확히 일치하는지 확인 / verify exactness at the nodes
y_at_nodes = lagrange_interpolant(x_data, y_data, x_data)
print('p_n(x_data) =', y_at_nodes)
print('y_data      =', y_data)
print('max |error| =', np.max(np.abs(y_at_nodes - y_data)))


`numpy.polyfit(deg=n)` 으로 같은 다항식을 얻을 수도 있다 &mdash; 표현은 다르지만 결과 다항식은 동일해야 한다 (보간 다항식은 유일하다).<br>
We can recover the same polynomial via `numpy.polyfit(deg=n)` &mdash; different *representation*, identical *polynomial* (the interpolating polynomial through $n+1$ distinct nodes is unique).


In [ ]:
coeffs = np.polyfit(x_data, y_data, deg=3)
y_polyfit = np.polyval(coeffs, x_dense)
print('max |Lagrange - polyfit| =', np.max(np.abs(y_dense - y_polyfit)))


## 룽게 현상<br>The Runge Phenomenon


작은 차수에서는 이렇게 잘 작동하니, 차수를 더 올리면 더 좋아질 것 같다. 절점을 등간격으로 더 촘촘히 잡으면 어떻게 될까?<br>
Small-degree fits worked beautifully, so surely cranking up the degree (more equispaced nodes) only makes things better. Right?

룽게(Carl Runge, 1901) 가 다음 함수에서 직접 보였다.<br>
Carl Runge (1901) showed otherwise on the following function:

$$
f(x) \;=\; \frac{1}{1 + 25\,x^2}, \qquad x \in [-1, 1].
$$

매끄럽고 별 문제 없어 보이지만, 등간격 절점에서 차수가 커지면 양 끝에서 *심한* 진동을 볼 수 있다.<br>
It looks tame and smooth &mdash; yet equispaced high-degree interpolation oscillates *wildly* near the ends.


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_dense = np.linspace(-1.0, 1.0, 800)
y_true = runge(x_dense)

plt.figure(figsize=(9, 5.5))
plt.plot(x_dense, y_true, 'k-', lw=2, label='$f(x)=1/(1+25x^2)$')

for n in (5, 10, 15, 20):
    x_nodes = np.linspace(-1.0, 1.0, n + 1)
    y_nodes = runge(x_nodes)
    y_interp = lagrange_interpolant(x_nodes, y_nodes, x_dense)
    plt.plot(x_dense, y_interp, label=f'$p_{{{n}}}$ (equispaced)')

plt.title('등간격 라그랑주 보간 / Equispaced Lagrange interpolation: Runge function')
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.ylim(-2.0, 2.0)
plt.legend(); plt.grid(True)
plt.show()


$n = 20$ 에서는 양 끝의 진동이 함수 자체보다 *훨씬* 크다. 차수를 올릴수록 *더* 나빠진다.<br>
At $n = 20$ the end oscillations are *larger* than the function itself. Raising the degree makes things *worse*, not better.

**왜 이렇게 되는가?**<br>
**Why does this happen?**

* 등간격 절점에서의 보간 다항식의 오차의 최대값은 *르베그 상수* $\Lambda_n$ 의 영향을 받는다. 등간격 절점에서는 $\Lambda_n \sim 2^n / (e\, n \log n)$ 으로 *지수적으로* 증가한다. 따라서 보간 대상이 되는 절점 값에 작은 진동 성분도 양 끝에서 거대한 다항식 진동으로 증폭된다.<br>
  The worst-case interpolation error is governed by the *Lebesgue constant* $\Lambda_n$. For equispaced nodes, $\Lambda_n \sim 2^n / (e n \log n)$ &mdash; it grows *exponentially* in $n$. Tiny perturbations in node values get amplified into huge polynomial oscillations near the endpoints.

* 마찬가지로, 자료점에서 만들어지는 반더몬데 행렬 $V_{ij} = x_i^{\,j}$ 의 조건수가 $n$ 과 함께 폭발한다. 같은 정보로 점점 더 *민감한* 시스템을 풀게 된다.<br>
  Equivalently, the Vandermonde matrix $V_{ij} = x_i^{\,j}$ becomes increasingly *ill-conditioned* &mdash; we are solving a more and more sensitive linear system from the same information.

따라서 단순히 차수를 올리는 것은 답이 아니다. 두 가지 길이 있다: (1) 절점을 다르게 고르거나, (2) 다항식을 *구간별* 로 잘게 쪼갠다 (다음 절에서 다룰 스플라인).<br>
So &ldquo;just use more nodes&rdquo; is not the fix. The two real escapes are: (1) pick smarter nodes, or (2) chop the interval into pieces, each fit by a *low-degree* polynomial (splines &mdash; the next notebook).


### 동적 탐색<br>Interactive Exploration


$n$ 을 천천히 올려 가며 양 끝에서 진동이 커지는 것을 관찰해 보자.<br>
Drag $n$ slowly upward and watch the end oscillations swallow the function.


In [ ]:
import os
from ipywidgets import interact, IntSlider

_ci = bool(os.getenv("CI", False))


def plot_runge_lagrange(n):
    x_dense = np.linspace(-1.0, 1.0, 800)
    y_true = runge(x_dense)

    x_nodes = np.linspace(-1.0, 1.0, n + 1)
    y_nodes = runge(x_nodes)
    y_interp = lagrange_interpolant(x_nodes, y_nodes, x_dense)

    plt.figure(figsize=(8, 4.5))
    plt.plot(x_dense, y_true, 'k-', lw=2, label='$f$')
    plt.plot(x_dense, y_interp, 'r-', label=f'$p_{{{n}}}$ (equispaced)')
    plt.plot(x_nodes, y_nodes, 'ko')

    err_inf = np.max(np.abs(y_interp - y_true))
    plt.title(f'n = {n}, $\|p_n - f\|_\infty$ = {err_inf:.3g}')
    plt.xlabel('$x$'); plt.ylabel('$y$')
    plt.ylim(-2.0, 2.0)
    plt.legend(); plt.grid(True)
    plt.show()


if _ci:
    # 위젯 대신 첫 단계만 렌더 / Render only the first step instead of using widget
    plot_runge_lagrange(10)
else:
    interact(
        plot_runge_lagrange,
        n=IntSlider(min=2, max=30, step=1, value=10, description='n :'),
    );


## 체비셰프 절점<br>Chebyshev Nodes


절점을 등간격이 아니라 양 끝에 더 *조밀하게* 잡으면 어떻게 될까? 체비셰프 절점은 다음과 같이 정의된다.<br>
What if we cluster nodes more *densely* near the endpoints instead of spacing them evenly? The Chebyshev nodes on $[-1, 1]$ are

$$
x_k \;=\; \cos\!\left(\frac{2k+1}{2(n+1)}\pi\right), \qquad k = 0, 1, \ldots, n.
$$

이 절점에서는 르베그 상수가 $\Lambda_n \sim \frac{2}{\pi} \log n$ 로 *훨씬* 천천히 자란다. 결과는 다음과 같다.<br>
For these nodes the Lebesgue constant grows only as $\Lambda_n \sim \frac{2}{\pi} \log n$ &mdash; *logarithmically*, not exponentially. Watch what happens:


In [ ]:
def chebyshev_nodes(n, a=-1.0, b=1.0):
    """체비셰프 절점 (n+1 개) 을 [a, b] 위에 생성.
       Generate n+1 Chebyshev nodes on [a, b]."""
    k = np.arange(n + 1)
    t = np.cos((2*k + 1) * np.pi / (2*(n + 1)))   # nodes on [-1, 1]
    return 0.5*(a + b) + 0.5*(b - a) * t


x_dense = np.linspace(-1.0, 1.0, 800)
y_true = runge(x_dense)

plt.figure(figsize=(9, 5.5))
plt.plot(x_dense, y_true, 'k-', lw=2, label='$f$')

for n in (10, 20):
    x_nodes = chebyshev_nodes(n)
    y_nodes = runge(x_nodes)
    y_interp = lagrange_interpolant(x_nodes, y_nodes, x_dense)
    plt.plot(x_dense, y_interp, label=f'$p_{{{n}}}$ (Chebyshev)')

plt.title('체비셰프 절점에서의 라그랑주 보간 / Lagrange interpolation at Chebyshev nodes')
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.ylim(-0.2, 1.2)
plt.legend(); plt.grid(True)
plt.show()


$n = 20$ 에서 등간격은 발산하지만, 체비셰프 절점은 함수와 거의 구분되지 않는다. *절점의 위치가 다항식 자체보다 더 중요할 수 있다.*<br>
At $n = 20$, equispaced nodes blow up; Chebyshev nodes are visually indistinguishable from $f$. *Node placement matters more than degree.*


## 르베그 상수에 관하여<br>About the Lebesgue Constant


방금 등간격 절점에서는 차수가 높아질수록 보간 결과가 *나빠지고*, 체비셰프 절점에서는 *좋아지는* 것을 보았다. 같은 라그랑주 공식, 같은 자료, 다른 절점 위치 &mdash; 결과가 극단적으로 갈리는 이유는 무엇인가? 정량적 답이 *르베그 상수* 이다.<br>
We just saw that with equispaced nodes the interpolation gets *worse* as the degree grows, while with Chebyshev nodes it gets *better*. Same Lagrange formula, same data, different node positions &mdash; why such opposite outcomes? The quantitative answer is the *Lebesgue constant*.

**르베그 함수 / Lebesgue function** &nbsp; 절점 $x_0, \ldots, x_n$ 이 정해지면 다음 함수가 정의된다.<br>
Once we fix nodes $x_0, \ldots, x_n$, define

$$
\lambda_n(x) \;=\; \sum_{j=0}^{n} \bigl|\, L_j(x) \,\bigr|.
$$

이 함수는 *자료값* $y_j$ 와 무관하다. 오직 절점의 *위치* 에만 의존한다. 절점 $x_i$ 에서는 $\lambda_n(x_i) = 1$ (왜?) 이고, 절점 사이에서는 $1$ 보다 클 수 있다.<br>
The Lebesgue function depends only on the *node positions*, not on the data values $y_j$. At the nodes themselves $\lambda_n(x_i) = 1$ (why?), and between nodes it can exceed $1$.

**르베그 상수 / Lebesgue constant** &nbsp; 르베그 함수의 구간 위 최대값을 르베그 상수라 한다.<br>
The Lebesgue constant is the maximum of the Lebesgue function over the interval:

$$
\Lambda_n \;=\; \max_{x \in [a, b]} \lambda_n(x).
$$


**왜 중요한가? 두 가지 해석.**<br>
**Why does it matter? Two interpretations.**

**(1) 최선 다항식 근사와의 비교 / Comparison with the best polynomial approximation.** 임의의 연속함수 $f$ 와 그 라그랑주 보간 $p_n$ 에 대해<br>
For any continuous $f$ and its Lagrange interpolant $p_n$,

$$
\| p_n - f \|_\infty \;\le\; (1 + \Lambda_n)\, E_n^*(f),
$$

여기서 $E_n^*(f)$ 는 *차수 $n$ 의 최선 다항식 근사* 의 오차이다 (와이어슈트라스의 정리에 의해 $f$ 가 연속이면 $n \to \infty$ 일 때 $0$ 으로 수렴함이 보장된다). 즉 $\Lambda_n$ 은 *보간이 최선 근사보다 얼마나 더 나쁠 수 있는지* 의 비율 한계를 알려준다.<br>
where $E_n^*(f)$ is the error of the *best* polynomial approximation of degree $n$ (Weierstrass guarantees this $\to 0$ as $n \to \infty$ for continuous $f$). So $\Lambda_n$ bounds *how much worse interpolation can be than the best possible polynomial fit*.

* $\Lambda_n$ 이 *작으면* (예: $\log n$ 정도) 라그랑주 보간은 최선 근사와 거의 차이가 없다. / If $\Lambda_n$ is *small* (e.g., $\sim \log n$), Lagrange interpolation is almost as good as the best approximation.
* $\Lambda_n$ 이 *크면* (예: $2^n$) 보간이 최선 근사보다 $2^n$ 배 더 나쁠 수 있다. 차수를 올린다고 해서 좋아지리라는 보장이 없어진다. / If $\Lambda_n$ is *large* (e.g., $2^n$), interpolation can be exponentially worse. Raising the degree no longer guarantees improvement.

**(2) 잡음 증폭률 / Noise amplification factor.** 실측 자료값 $y_j$ 가 진짜 값과 $\varepsilon_j$ 만큼 차이가 나고 $|\varepsilon_j| \le \varepsilon$ 이라 하자. 그러면 보간 다항식의 오차도 다음과 같이 제한된다.<br>
Suppose the data values $y_j$ are perturbed from their true values by errors $\varepsilon_j$ with $|\varepsilon_j| \le \varepsilon$. The resulting perturbation in the interpolant satisfies

$$
\| \tilde p_n - p_n \|_\infty \;\le\; \Lambda_n \cdot \varepsilon.
$$

따라서 $\Lambda_n$ 은 자료의 측정 오차가 보간 결과로 *증폭* 되는 비율이기도 하다. $\Lambda_n = 10^4$ 이면 자료 오차의 $10^4$ 배까지 다항식이 흔들릴 수 있다.<br>
So $\Lambda_n$ is also the *amplification factor* by which measurement noise in the data shows up in the interpolant. A Lebesgue constant of $10^4$ means data noise can be magnified by $10^4$&times; into the polynomial.


In [ ]:
# 르베그 상수를 직접 계산 / compute the Lebesgue constant numerically
def lebesgue_constant(x_nodes, n_sample=2001):
    # L_n = max over a fine grid of sum_j |L_j(x)|
    x_grid = np.linspace(x_nodes[0], x_nodes[-1], n_sample)
    lam = np.zeros_like(x_grid)
    for j in range(len(x_nodes)):
        lam += np.abs(lagrange_basis(x_nodes, j, x_grid))
    return lam.max()


n_values = np.arange(4, 31)
lam_equi = np.array([lebesgue_constant(np.linspace(-1.0, 1.0, n + 1)) for n in n_values])
lam_cheb = np.array([lebesgue_constant(chebyshev_nodes(n))            for n in n_values])

# 점근식 비교선 / asymptotic reference curves
lam_equi_asym = 2.0**(n_values + 1) / (np.e * n_values * np.log(np.maximum(n_values, 2)))
lam_cheb_asym = (2.0 / np.pi) * np.log(n_values + 1) + 0.9625      # +bias

plt.figure(figsize=(7, 5))
plt.semilogy(n_values, lam_equi, 'ro-', label='equispaced (computed)')
plt.semilogy(n_values, lam_cheb, 'bs-', label='Chebyshev (computed)')
plt.semilogy(n_values, lam_equi_asym, 'r:',  alpha=0.6, label=r'$\sim 2^{n+1}/(e\, n\, \log n)$')
plt.semilogy(n_values, lam_cheb_asym, 'b:',  alpha=0.6, label=r'$\sim (2/\pi)\log(n+1)$')
plt.xlabel('$n$  (degree of the interpolant)')
plt.ylabel(r'$\Lambda_n$')
plt.title(r'르베그 상수 / Lebesgue constant: equispaced vs Chebyshev')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()


**그림에서 읽을 수 있는 것 / What the plot shows.**<br>

* 등간격 절점에서 $\Lambda_n$ 은 **지수적** 으로 증가한다 ($n = 30$ 에서 이미 $10^7$ 을 넘는다). 차수를 한 단계 올릴 때마다 잡음 증폭률이 약 $2$ 배가 된다는 뜻이다. 룽게 현상은 단순한 우연이 아닌 *구조적* 결과임을 보여준다.<br>
  For equispaced nodes, $\Lambda_n$ grows **exponentially** &mdash; already past $10^7$ at $n = 30$. Each added degree roughly doubles the noise amplification. The Runge phenomenon is not a coincidence; it is a *structural* consequence of this exponential blow-up.
* 체비셰프 절점에서는 $\Lambda_n$ 이 **로그적** 으로만 증가한다 ($n = 30$ 에서도 약 $3$ 정도). 잡음 증폭률이 거의 변하지 않는다.<br>
  For Chebyshev nodes, $\Lambda_n$ grows only **logarithmically** &mdash; about $3$ at $n = 30$. Noise amplification barely moves.
* 1958 년 Erd&ouml;s 는 *어떤 절점 선택에 대해서도* $\Lambda_n \ge \frac{2}{\pi}\log(n+1) + C$ 라는 보편 하한을 보였다. 즉 *체비셰프 절점은 점근적으로 최적이다* &mdash; 어떤 절점 배치도 로그 증가율보다 더 좋을 수 없다.<br>
  Erd&ouml;s (1958) proved that for *any* node placement, $\Lambda_n \ge \frac{2}{\pi}\log(n+1) + C$. So *Chebyshev nodes achieve the optimal asymptotic rate* &mdash; no node placement can do better than logarithmic growth.

**요약.** 등간격 절점에서 라그랑주 보간이 폭발하는 것은 *알고리즘 자체의 결함이 아니라 절점의 선택* 이 만든 문제이다. 절점 위치만 바꿔도 같은 라그랑주 공식이 안정적인 방법이 된다.<br>
**Summary.** High-degree Lagrange interpolation on equispaced nodes is not bad because the *algorithm* is bad &mdash; it is bad because the *node placement* is bad. Change the node placement and the same Lagrange formula becomes a stable method.


## 반더몬데 행렬<br>The Vandermonde Matrix


르베그 상수가 *함수해석학의 시각* 에서 이 폭발을 정량화한다면, *선형대수의 시각* 에서 본질적으로 같은 결론에 이르는 길이 있다 &mdash; 반더몬데 행렬이다.<br>
The Lebesgue constant explains the blow-up from a *functional-analysis* point of view. There is a parallel route from *linear algebra* &mdash; the Vandermonde matrix.

**가장 직접적인 표현 / The most direct representation.** &nbsp; 라그랑주 기저를 쓰지 않고, 보간 다항식을 단항식 (monomial) 의 선형결합으로 쓰자.<br>
Without using the Lagrange basis, write the interpolant as a linear combination of monomials:

$$
p_n(x) \;=\; a_0 + a_1 x + a_2 x^2 + \cdots + a_n x^n.
$$

각 절점에서 보간 조건 $p_n(x_i) = y_i$ 를 적용하면 다음의 *선형 연립방정식* 을 얻는다.<br>
Imposing the interpolation conditions $p_n(x_i) = y_i$ at every node yields a *linear system*:

$$
\underbrace{\begin{pmatrix}
1 & x_0 & x_0^2 & \cdots & x_0^n \\
1 & x_1 & x_1^2 & \cdots & x_1^n \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x_n & x_n^2 & \cdots & x_n^n
\end{pmatrix}}_{V \;\text{(Vandermonde matrix)}}
\begin{pmatrix} a_0 \\ a_1 \\ \vdots \\ a_n \end{pmatrix}
\;=\;
\begin{pmatrix} y_0 \\ y_1 \\ \vdots \\ y_n \end{pmatrix}.
$$

이 행렬을 **반더몬데 행렬** 이라 한다. 원소는 단순히 $V_{ij} = x_i^{\,j}$ 이다 ($i, j = 0, 1, \ldots, n$). 절점이 모두 다르면 $V$ 는 가역이므로 $a = V^{-1} y$ 로 *유일한* 다항식 계수를 얻는다.<br>
This matrix $V$, with entries $V_{ij} = x_i^{\,j}$, is the **Vandermonde matrix**. As long as the nodes are distinct, $V$ is invertible and $a = V^{-1} y$ gives the (unique) polynomial coefficients.


**작은 예 / Small example.** &nbsp; 4개의 자료점 $x = (0, 1, 2, 4)$, $y = (1, 2, 0.5, 3)$ 에 대한 반더몬데 행렬을 직접 보자.<br>
Let's actually look at the Vandermonde matrix for a small example: $x = (0, 1, 2, 4)$, $y = (1, 2, 0.5, 3)$.


In [ ]:
x_data = np.array([0.0, 1.0, 2.0, 4.0])
y_data = np.array([1.0, 2.0, 0.5, 3.0])

# numpy.vander(x, increasing=True) 는 V[i, j] = x_i^j
# V[i, j] = x_i^j with the increasing-power convention
V = np.vander(x_data, increasing=True)
print('V =')
print(V)

# 보간 다항식의 계수 / interpolation polynomial coefficients
a = np.linalg.solve(V, y_data)
print('\na (coefficients of 1, x, x^2, x^3) =', a)

# 검증 / verify against the Lagrange interpolant we built earlier
x_dense = np.linspace(-0.5, 4.5, 200)
y_mono = sum(a[k] * x_dense**k for k in range(len(a)))
y_lagr = lagrange_interpolant(x_data, y_data, x_dense)
print('\nmax |Vandermonde - Lagrange| =', np.max(np.abs(y_mono - y_lagr)))


두 표현 (반더몬데 vs 라그랑주) 은 *같은* 다항식을 다른 *기저* 로 적었을 뿐이다. 보간 다항식이 유일하기 때문에 결과는 일치한다.<br>
The two representations (Vandermonde vs Lagrange) are the *same* polynomial written in different *bases*. The interpolant is unique, so the results agree.


**시각화: 행렬의 구조 / Visualizing the matrix structure.** &nbsp; $n + 1 = 9$ 개의 등간격 절점에서 반더몬데 행렬의 절댓값을 색깔로 그려 보자. 색의 *동적 범위* 가 핵심이다.<br>
Let's visualize the matrix entries (in absolute value) for 9 equispaced nodes on $[-1, 1]$. The key thing to look at is the *dynamic range* of the colors.

같은 행렬을 등간격 $[0, 1]$ 로 다시 그려 보자. 한쪽 끝의 절점들이 $x_i^{\,n}$ 항에서 거의 $1$ 인 동시에 다른 한쪽 끝의 절점들은 거의 $0$ 인 점에 주의.<br>
Then again on $[0, 1]$ &mdash; note that nodes at one end give $x_i^n \approx 1$ while nodes at the other end give $x_i^n \approx 0$.


In [ ]:
from matplotlib.colors import LogNorm

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, (a_, b_) in zip(axes, [(-1.0, 1.0), (0.0, 1.0)]):
    x_nodes = np.linspace(a_, b_, 9)
    V = np.vander(x_nodes, increasing=True)
    # 0 을 LogNorm 에 넣을 수 없으므로 미세값으로 대체 / clip zero for log scale
    im = ax.matshow(np.abs(V) + 1e-12, norm=LogNorm(vmin=1e-6, vmax=1.0), cmap='viridis')
    ax.set_title(f'$|V_{{ij}}|$ on  $x \\in [{a_}, {b_}]$,  $n+1 = 9$')
    ax.set_xlabel('column $j$  (power)')
    ax.set_ylabel('row $i$  (node)')
    plt.colorbar(im, ax=ax, label='$|V_{ij}|$ (log scale)')

plt.tight_layout()
plt.show()


한 행렬의 원소들이 거의 *0* 부터 *1* 까지 (그리고 음의 절점에서는 부호도 바뀐) 여러 자릿수에 걸쳐 분포하는 것을 볼 수 있다. 이런 *동적 범위* 가 큰 행렬은 일반적으로 풀기가 까다롭다 &mdash; 작은 원소가 큰 원소에 묻혀 정보가 손실된다.<br>
The matrix entries span many orders of magnitude from near-$0$ to $1$ (and at negative-$x$ nodes, sign-flipping too). Matrices with this kind of *dynamic range* are notoriously hard to solve &mdash; small entries get drowned out by large ones, and information is lost.


**정량화: 조건수 / Quantifying it: the condition number.** &nbsp; 선형 시스템 $Va = y$ 에서 우변에 작은 오차 $\delta y$ 가 있으면 해 $a$ 의 오차는 다음과 같이 제한된다.<br>
For the linear system $Va = y$, small perturbations $\delta y$ in the right-hand side produce solution errors bounded by

$$
\frac{\| \delta a \|}{\| a \|} \;\le\; \kappa(V) \cdot \frac{\| \delta y \|}{\| y \|},
\qquad
\kappa(V) \;=\; \| V \| \cdot \| V^{-1} \|.
$$

여기서 $\kappa(V)$ 가 *조건수* 이다. 조건수가 $10^k$ 정도이면 풀이 과정에서 약 $k$ 자리의 정밀도가 손실될 수 있다.<br>
Here $\kappa(V)$ is the *condition number*. A condition number of $10^k$ means roughly $k$ digits of precision can be lost during the solve.

이제 절점을 등간격 vs 체비셰프로 두고 $\kappa(V)$ 가 $n$ 과 함께 어떻게 변하는지 직접 보자.<br>
Let's see how $\kappa(V)$ grows with $n$ for equispaced vs Chebyshev nodes.


In [ ]:
def vandermonde_cond(x_nodes):
    return np.linalg.cond(np.vander(x_nodes, increasing=True))


n_values = np.arange(4, 31)
cond_equi = np.array([vandermonde_cond(np.linspace(-1.0, 1.0, n + 1)) for n in n_values])
cond_cheb = np.array([vandermonde_cond(chebyshev_nodes(n))            for n in n_values])

plt.figure(figsize=(7, 5))
plt.semilogy(n_values, cond_equi, 'ro-', label='equispaced')
plt.semilogy(n_values, cond_cheb, 'bs-', label='Chebyshev')
plt.axhline(1.0/np.finfo(float).eps, color='k', linestyle=':', alpha=0.6,
            label='float64 precision wall ($1/\\epsilon$)')
plt.xlabel('$n$')
plt.ylabel(r'$\kappa(V)$  (condition number)')
plt.title('반더몬데 행렬의 조건수 / Condition number of the Vandermonde matrix')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()


**그림에서 읽을 수 있는 것 / What the plot shows.**<br>

* 등간격 절점에서 $\kappa(V)$ 는 *지수적* 으로 증가한다. $n \approx 18$ 부근에서 이미 부동소수점 정밀도의 벽 ($\sim 10^{16}$) 에 도달한다 &mdash; 이 지점부터는 $V a = y$ 를 풀어도 *모든 자릿수* 가 잡음일 수 있다.<br>
  For equispaced nodes, $\kappa(V)$ grows *exponentially*. By $n \approx 18$ it already hits the double-precision wall ($\sim 10^{16}$) &mdash; past this point, solving $Va = y$ can return a result whose every digit is noise.
* 체비셰프 절점에서는 $\kappa(V)$ 가 훨씬 천천히 증가한다. 같은 $n$ 에서 등간격보다 수십 자릿수 더 안정적이다.<br>
  For Chebyshev nodes, $\kappa(V)$ grows much more slowly &mdash; tens of orders of magnitude better than equispaced at the same $n$.

**르베그 상수와의 연결 / Connection back to the Lebesgue constant.** &nbsp; 두 그림의 *모양* 이 닮은 것은 우연이 아니다. $\Lambda_n$ 과 $\kappa(V)$ 모두 같은 *근원* 에서 비롯된다 &mdash; 등간격 절점에서 다항식 보간 문제 그 자체가 *조건이 나쁘다*. 어떤 표현 (라그랑주 / 단항식) 을 쓰든, 어떤 알고리즘으로 풀든, 이 조건성은 피할 수 없다.<br>
The two plots have similar *shapes* &mdash; not by coincidence. $\Lambda_n$ and $\kappa(V)$ both come from the same root: on equispaced nodes, the polynomial interpolation problem itself is *ill-conditioned*. No representation (Lagrange / monomial) and no algorithm can paper over this.

**실용적 결론 / Practical takeaway.** &nbsp; 만약 $n$ 이 크고 절점을 *직접 고를 수 있다면* 항상 체비셰프 절점을 사용하라. 절점이 외부에서 정해져 있다면 (실험 자료 등) 단일 고차 다항식 보간 자체를 포기하고 *구간별 다항식* (다음 절의 스플라인) 을 쓰라.<br>
**Practical takeaway:** if $n$ is large and you control the node placement, always use Chebyshev nodes. If the nodes are fixed by the data (measured points), abandon single high-degree polynomial interpolation altogether and switch to *piecewise* polynomials (the splines of the next section).


## 언제 라그랑주 내삽을 쓰는가<br>When to Use Lagrange Interpolation


| 상황<br>Situation | 추천<br>Recommendation |
|---|---|
| 자료점이 적음 ($n \lesssim 5$), 등간격<br>Few points ($n \lesssim 5$), equispaced | 라그랑주 OK |
| 자료점이 많음, 절점을 *고를 수 있음*<br>Many points, nodes are *yours to choose* | 라그랑주 + 체비셰프 절점 |
| 자료점이 많음, 절점이 *고정* (보통 측정값)<br>Many points, nodes are *fixed* (usually measurements) | 스플라인 (다음 절) / Splines (next notebook) |
| 미분이 매끄럽게 이어져야 함<br>Need a smooth derivative through the data | 3차 스플라인 / Cubic spline |

요약하면, 라그랑주는 *절점을 자유롭게 고를 수 있는* 함수 근사에는 강력하지만, 임의의 자료에 대한 일반적인 수학적 도구로는 한계가 있다. 다음 절에서 이 한계를 우회하는 *구간별 다항식* 인 스플라인을 다룬다.<br>
In short: Lagrange is powerful for *function approximation when nodes are yours to design*, but it is not the right hammer for arbitrary data. The next notebook covers the workhorse that escapes these limitations &mdash; *piecewise* polynomials, a.k.a. splines.


## 연습 문제<br>Exercises


Try this 1: $\sin\theta^\circ$ 를 $\theta = 0, 30, 60, \ldots, 180^\circ$ 에서 자료점으로 두고 라그랑주 다항식을 만들어 보시오. $\theta = 1^\circ$ 단위로 평가하여 정확한 $\sin$ 과 비교하시오.<br>
Build a Lagrange interpolant from $\sin\theta^\circ$ at $\theta = 0, 30, 60, \ldots, 180^\circ$. Evaluate it on a $1^\circ$ grid and compare with the true $\sin$ values.


Try this 2: 위 예제에서 $\theta$ 의 절점을 1, 11, 21, ..., $171^\circ$ 등으로 *비등간격* 으로 바꾸어 보시오. 결과가 어떻게 달라지는가?<br>
Repeat the previous exercise with *non-equispaced* nodes ($\theta = 1, 11, 21, \ldots, 171^\circ$). How does the result change?


Try this 3: $f(x) = e^x$ 를 $[-1, 1]$ 위에서 등간격 5개 절점과 체비셰프 5개 절점으로 보간해 보시오. 두 결과의 최대 오차 차이는 얼마인가?<br>
Interpolate $f(x) = e^x$ on $[-1, 1]$ using 5 equispaced nodes and 5 Chebyshev nodes. What is the difference in maximum error?


## 참고문헌<br>References


* R. L. Burden, J. D. Faires, A. M. Burden, *Numerical Analysis*, 10th Ed., Cengage, 2016 (Ch. 3 *Interpolation and Polynomial Approximation*).
* L. N. Trefethen, *Approximation Theory and Approximation Practice*, SIAM, 2013 (Ch. 5 *Barycentric interpolation formula*; Ch. 15 *Lebesgue constants*).
* C. Runge, &ldquo;&Uuml;ber empirische Funktionen und die Interpolation zwischen &auml;quidistanten Ordinaten,&rdquo; *Zeitschrift f&uuml;r Mathematik und Physik*, vol. 46, pp. 224&ndash;243, 1901.


## Final Bell<br>마지막 종


In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");
